### Tools

Models can request to call tools that perform tasks such as fetching data form a database ,searching the web or running code. Tools are pairing of:

     1. A schema,including the name of the tool, a description, and/or argument definitions (often a JSON schema)
     2. A function or coroutine to execute.

In [2]:
from dotenv import load_dotenv
load_dotenv()

import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("where is eiffel tower?")
response

AIMessage(content="<think>\nOkay, the user is asking where the Eiffel Tower is located. Let me start by recalling that the Eiffel Tower is one of the most famous landmarks in the world. I know it's in France, but I need to be more specific. The city is Paris, right? But which part of Paris? I think it's situated on the Champ de Mars. Let me confirm that.\n\nI remember reading that it was built for the 1889 World's Fair, which was to celebrate the centennial of the French Revolution. The location in Paris was chosen to showcase France's industrial progress. The Eiffel Tower is near the Seine River. There's a bridge called Pont d'Iéna that connects it to the Trocadéro Palace. So, putting it all together, the Eiffel Tower is in Paris, France, on the Champ de Mars, near the Seine River, close to the Trocadéro area.\n\nWait, is the Champ de Mars part of the 7th arrondissement? I should check that. Yes, the Champ de Mars is in the 7th arrondissement of Paris. The Eiffel Tower itself is a par

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"it is sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("what is the weather in paris?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Paris. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Paris here. I should structure the tool call with the location parameter set to Paris. Make sure the JSON is correctly formatted.\n', 'tool_calls': [{'id': 'xaf6k0xed', 'function': {'arguments': '{"location":"Paris"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 153, 'total_tokens': 237, 'completion_time': 0.141438918, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.006053692, 'prompt_tokens_details': None, 'queue_time': 0.280806292, 'total_time': 0.14749261}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019eb71c-d6a8-7f91-a047-a6d9d3f

### Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "what is the weather in paris?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tool and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weather in paris is sunny.

The current weather in Paris is as follows:
- **Temperature**: 22°C
- **Humidity**: 65%
- **Conditions**: Partly cloudy

Let me know if you'd like additional details! ☀️
